In [ ]:
import pandas as pd

data= pd.read_csv('2025_LoL_esports_match_data_from_OraclesElixir.csv')

data

             gameid datacompleteness  url league  year   split  playoffs  \
0  LOLTMNT03_179647         complete  NaN   LFL2  2025  Winter         0   
1  LOLTMNT03_179647         complete  NaN   LFL2  2025  Winter         0   
2  LOLTMNT03_179647         complete  NaN   LFL2  2025  Winter         0   
3  LOLTMNT03_179647         complete  NaN   LFL2  2025  Winter         0   
4  LOLTMNT03_179647         complete  NaN   LFL2  2025  Winter         0   

                  date  game  patch  ...  opp_csat25 golddiffat25 xpdiffat25  \
0  2025-01-11 11:11:24     1  15.01  ...       200.0        224.0       -1.0   
1  2025-01-11 11:11:24     1  15.01  ...       157.0      -2363.0    -1444.0   
2  2025-01-11 11:11:24     1  15.01  ...       241.0      -1552.0    -2465.0   
3  2025-01-11 11:11:24     1  15.01  ...       257.0      -2613.0    -1156.0   
4  2025-01-11 11:11:24     1  15.01  ...        20.0       -662.0     -734.0   

  csdiffat25 killsat25 assistsat25 deathsat25  opp_killsat25 o

C:\Users\JUAN BERNAL\AppData\Local\Temp\ipykernel_14416\1589958307.py:3: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  data= pd.read_csv('2025_LoL_esports_match_data_from_OraclesElixir.csv')


In [ ]:
print("Valores únicos en 'league':", data['league'].nunique())
print(data['league'].unique())

# Modelo de Apuestas LCK 2025
## Pipeline completo: Feature Engineering → ML → Kelly Criterion

---
## Sección 1 — Imports y Filtrado LCK

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, log_loss, accuracy_score,
                             brier_score_loss, RocCurveDisplay)
import xgboost as xgb
import lightgbm as lgb

pd.set_option('display.max_columns', 50)
RANDOM_SEED = 42

lck = data[data['league'] == 'LCK'].copy()
lck['date'] = pd.to_datetime(lck['date'])

team_rows = lck[lck['position'] == 'team'].copy()
player_rows = lck[lck['position'] != 'team'].copy()

print(f"Partidas LCK únicas: {team_rows['gameid'].nunique()}")
print(f"Splits: {sorted(team_rows['split'].unique())}")
print(f"Equipos: {sorted(team_rows['teamname'].unique())}")
print(f"Rango de fechas: {team_rows['date'].min().date()} → {team_rows['date'].max().date()}")
print(f"Patches: {sorted(team_rows['patch'].unique())}")

---
## Sección 2 — Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LCK 2025 — Análisis Exploratorio', fontsize=14, fontweight='bold')

# Win rate por equipo
team_wr = team_rows.groupby('teamname')['result'].mean().sort_values()
axes[0].barh(team_wr.index, team_wr.values, color=['#d9534f' if v < 0.5 else '#5cb85c' for v in team_wr.values])
axes[0].axvline(0.5, ls='--', color='gray', alpha=0.7)
axes[0].set_xlabel('Win Rate')
axes[0].set_title('Win Rate por Equipo')

# Win rate Blue vs Red
side_wr = team_rows.groupby('side')['result'].agg(['mean', 'count'])
bars = axes[1].bar(side_wr.index, side_wr['mean'], color=['royalblue', 'firebrick'], width=0.5)
axes[1].set_ylim(0.45, 0.56)
axes[1].axhline(0.5, ls='--', color='gray', alpha=0.7)
axes[1].set_title('Win Rate por Lado')
axes[1].set_ylabel('Win Rate')
for bar, (_, row) in zip(bars, side_wr.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f"{row['mean']:.1%}\n(n={int(row['count'])})", ha='center', fontsize=10)

# Win rate por split
split_order = ['Cup', 'Rounds 1-2', 'Rounds 3-5']
split_side = team_rows.groupby(['split', 'side'])['result'].mean().unstack()
split_side = split_side.reindex(split_order)
split_side.plot(kind='bar', ax=axes[2], color=['royalblue', 'firebrick'], rot=20)
axes[2].axhline(0.5, ls='--', color='gray', alpha=0.7)
axes[2].set_title('Win Rate por Split y Lado')
axes[2].set_ylabel('Win Rate')
axes[2].legend(title='Lado')

plt.tight_layout()
plt.show()

# Duración promedio de partidas
print(f"\nDuración promedio: {team_rows['gamelength'].mean()/60:.1f} min")
print(f"Duración mínima:   {team_rows['gamelength'].min()/60:.1f} min")
print(f"Duración máxima:   {team_rows['gamelength'].max()/60:.1f} min")

In [ ]:
# Correlación de features con resultado (para entender qué predice ganar)
CORR_FEATURES = ['kills', 'deaths', 'dragons', 'barons', 'towers',
                 'golddiffat15', 'golddiffat25', 'gspd', 'earned gpm',
                 'visionscore', 'ckpm', 'void_grubs', 'firstbaron', 'firstdragon']

# Rellenar NaN en golddiffat25 y void_grubs
team_rows['golddiffat25'] = team_rows['golddiffat25'].fillna(0)
team_rows['void_grubs']   = team_rows['void_grubs'].fillna(0)
team_rows['atakhans']     = team_rows['atakhans'].fillna(0)

corr = team_rows[CORR_FEATURES + ['result']].corr()['result'].drop('result').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#5cb85c' if v > 0 else '#d9534f' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlación de Pearson con resultado')
plt.title('Correlación de Features con Victoria (LCK 2025)\n⚠️  Features in-game (en rojo) no se pueden usar para apuestas pre-partida')
plt.tight_layout()
plt.show()

print("\nTop correlaciones:")
print(corr.round(3))

---
## Sección 3 — Feature Engineering (Pre-game, sin data leakage)

**Regla fundamental:** Solo se usan features conocidas ANTES de que empiece la partida.  
`shift(1)` garantiza que cada fila usa únicamente el historial de partidas anteriores.

In [ ]:
# Ordenar cronológicamente
team_rows = team_rows.sort_values('date').reset_index(drop=True)

# --- Rolling features con shift(1) para evitar leakage ---
ROLLING_COLS = [
    'result', 'golddiffat15', 'golddiffat25', 'dragons', 'barons',
    'towers', 'earned gpm', 'gspd', 'visionscore', 'ckpm', 'void_grubs'
]
WINDOWS = [3, 5, 10]

def make_rolling_features(df, cols, windows):
    """Rolling mean por equipo con shift(1) — sin leakage."""
    for col in cols:
        for w in windows:
            df[f'{col}_roll{w}'] = df.groupby('teamname')[col].transform(
                lambda x: x.shift(1).rolling(w, min_periods=max(1, w//2)).mean()
            )
    return df

team_rows = make_rolling_features(team_rows, ROLLING_COLS, WINDOWS)

# --- Feature de racha (win streak) ---
def compute_streak(series):
    streak, current = [], 0
    for val in series.shift(1):
        if pd.isna(val):
            streak.append(np.nan); current = 0
        elif val == 1:
            current = max(1, current + 1); streak.append(current)
        else:
            current = min(-1, current - 1); streak.append(current)
    return pd.Series(streak, index=series.index)

team_rows['win_streak'] = team_rows.groupby('teamname')['result'].transform(compute_streak)

# --- Días desde última partida (descanso/fatiga) ---
team_rows['days_since_last'] = team_rows.groupby('teamname')['date'].transform(
    lambda x: x.diff().dt.days
)

# --- Win rate acumulado de la temporada ---
team_rows['cumulative_wr'] = team_rows.groupby('teamname')['result'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# --- Features de meta ---
patch_order = sorted(team_rows['patch'].unique())
team_rows['patch_num']  = team_rows['patch'].map({p: i for i, p in enumerate(patch_order)})
team_rows['is_blue']    = (team_rows['side'] == 'Blue').astype(int)
team_rows['is_playoffs'] = team_rows['playoffs'].astype(int)
split_map = {'Cup': 0, 'Rounds 1-2': 1, 'Rounds 3-5': 2}
team_rows['split_num']  = team_rows['split'].map(split_map)

roll_feats = [c for c in team_rows.columns if '_roll' in c]
print(f"Features rolling creadas: {len(roll_feats)}")
print(f"Ejemplo: {roll_feats[:6]}")

---
## Sección 4 — Dataset Match-Level (1 fila por partida)

In [ ]:
PRE_GAME_FEATS = (
    [f'{c}_roll{w}' for c in ROLLING_COLS for w in WINDOWS] +
    ['win_streak', 'days_since_last', 'cumulative_wr']
)
META_FEATS = ['patch_num', 'is_playoffs', 'split_num']

def build_match_df(df, pregame_feats, meta_feats):
    """Pivota a 1 fila por partida con columnas blue_ / red_ y diferenciales diff_."""
    blue = df[df['side'] == 'Blue'].copy()
    red  = df[df['side'] == 'Red'].copy()

    blue_rename = {f: f'blue_{f}' for f in pregame_feats}
    red_rename  = {f: f'red_{f}'  for f in pregame_feats}

    blue_sel = (blue[['gameid', 'date', 'split', 'teamname', 'result', 'is_blue']
                     + pregame_feats + meta_feats]
                .rename(columns={**{'teamname': 'blue_team', 'result': 'blue_result'},
                                 **blue_rename}))

    red_sel = (red[['gameid', 'teamname', 'result'] + pregame_feats]
               .rename(columns={**{'teamname': 'red_team', 'result': 'red_result'},
                                **red_rename}))

    match_df = blue_sel.merge(red_sel, on='gameid').sort_values('date').reset_index(drop=True)

    # Target: ¿ganó el equipo Blue?
    match_df['target'] = match_df['blue_result']

    # Diferenciales Blue - Red (más predictivos que valores absolutos)
    for f in pregame_feats:
        bf, rf = f'blue_{f}', f'red_{f}'
        if bf in match_df.columns and rf in match_df.columns:
            match_df[f'diff_{f}'] = match_df[bf] - match_df[rf]

    return match_df

match_df = build_match_df(team_rows, PRE_GAME_FEATS, META_FEATS)

# Imputar NaN (partidas iniciales sin historial) con media global
feature_cols = [c for c in match_df.columns
                if c.startswith(('blue_', 'red_', 'diff_'))
                and c not in ['blue_team', 'red_team', 'blue_result', 'red_result']]
for col in feature_cols:
    match_df[col] = match_df[col].fillna(match_df[col].mean())

print(f"Dataset match-level: {match_df.shape}")
print(f"NaN restantes: {match_df[feature_cols].isna().sum().sum()}")
print(f"\nDistribución de splits:")
print(match_df['split'].value_counts())

---
## Sección 5 — Modelos con Time-Aware Cross-Validation

⚠️ **k-fold estándar está PROHIBIDO** — usaría datos futuros para predecir el pasado (data leakage temporal).  
Se usan splits cronológicos expandibles:
- **Fold 1:** Train = Cup → Test = Rounds 1-2  
- **Fold 2:** Train = Cup + Rounds 1-2 → Test = Rounds 3-5

In [ ]:
# --- Definir splits temporales ---
mask_cup = match_df['split'] == 'Cup'
mask_r12 = match_df['split'] == 'Rounds 1-2'
mask_r35 = match_df['split'] == 'Rounds 3-5'

X = match_df[feature_cols]
y = match_df['target']

X_cup, y_cup = X[mask_cup], y[mask_cup]
X_r12, y_r12 = X[mask_r12], y[mask_r12]
X_r35, y_r35 = X[mask_r35], y[mask_r35]

print(f"Cup (train base):  {len(X_cup)} partidas")
print(f"Rounds 1-2 (fold1 test): {len(X_r12)} partidas")
print(f"Rounds 3-5 (fold2 test): {len(X_r35)} partidas")

# --- Definir modelos ---
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_SEED))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_leaf=10,
        random_state=RANDOM_SEED, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        num_leaves=15, min_child_samples=15,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, verbose=-1
    ),
}

# --- Entrenar y evaluar ---
def evaluate(model, name, X_train, y_train, X_test, y_test, fold_name):
    if 'XGBoost' in name:
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)], verbose=False)
    else:
        model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        'Modelo': name, 'Fold': fold_name,
        'Accuracy': accuracy_score(y_test, preds),
        'ROC-AUC':  roc_auc_score(y_test, proba),
        'Log Loss': log_loss(y_test, proba),
        'Brier':    brier_score_loss(y_test, proba),
    }

results = []
X_train_f2 = pd.concat([X_cup, X_r12])
y_train_f2 = pd.concat([y_cup, y_r12])

for name, model in models.items():
    results.append(evaluate(model, name, X_cup, y_cup, X_r12, y_r12, 'Cup→R1-2'))

# Re-instanciar para fold 2
models2 = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_SEED))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=10, random_state=RANDOM_SEED, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=2.0, random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0),
    'LightGBM': lgb.LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.03, num_leaves=15, min_child_samples=15, reg_alpha=1.0, reg_lambda=2.0, random_state=RANDOM_SEED, verbose=-1),
}
for name, model in models2.items():
    results.append(evaluate(model, name, X_train_f2, y_train_f2, X_r35, y_r35, 'Cup+R1-2→R3-5'))

results_df = pd.DataFrame(results)

# Tabla comparativa
pivot = results_df.pivot_table(index='Modelo', columns='Fold',
        values=['Accuracy', 'ROC-AUC', 'Log Loss']).round(3)
print("\n=== Resultados por Modelo y Fold ===")
print(pivot.to_string())

# Guardar mejor modelo (fold 2, mayor AUC)
best_model_name = results_df[results_df['Fold'] == 'Cup+R1-2→R3-5'].sort_values('ROC-AUC', ascending=False).iloc[0]['Modelo']
print(f"\nMejor modelo (Fold 2): {best_model_name}")

In [ ]:
# Gráfico de comparación de modelos
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de Modelos — LCK Betting Model', fontsize=13, fontweight='bold')

metrics = ['Accuracy', 'ROC-AUC', 'Log Loss']
folds = ['Cup→R1-2', 'Cup+R1-2→R3-5']
colors = ['#4878cf', '#6acc65']

for ax, metric in zip(axes, metrics):
    fold_data = results_df.groupby(['Fold', 'Modelo'])[metric].first().unstack('Fold')
    fold_data[folds].plot(kind='bar', ax=ax, color=colors, rot=25, legend=(metric == 'Accuracy'))
    ax.set_title(metric)
    ax.set_xlabel('')
    if metric != 'Log Loss':
        ax.axhline(0.5, ls='--', color='gray', alpha=0.5, label='Baseline')
    ax.set_ylim(ax.get_ylim()[0] * 0.95, ax.get_ylim()[1] * 1.05)

plt.tight_layout()
plt.show()

---
## Sección 6 — Calibración de Probabilidades

Para apuestas, la **calidad de las probabilidades importa más que el accuracy**.  
Un modelo que dice "70% de probabilidad" debe ganar ~70% de las veces.  
Se usa Platt scaling (sigmoid) con `CalibratedClassifierCV`.

In [ ]:
# Calibrar el mejor modelo del fold 2
best_base = models2[best_model_name]
calibrated_model = CalibratedClassifierCV(best_base, method='sigmoid', cv=5)
calibrated_model.fit(X_train_f2, y_train_f2)
cal_proba = calibrated_model.predict_proba(X_r35)[:, 1]
raw_proba = models2[best_model_name].predict_proba(X_r35)[:, 1]

# Curva de calibración (reliability diagram)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Calibración de Probabilidades — {best_model_name}', fontsize=13)

# Reliability diagram
for proba, label, style in [(raw_proba, 'Sin calibrar', 's--'), (cal_proba, 'Calibrado (Platt)', 'o-')]:
    frac_pos, mean_pred = calibration_curve(y_r35, proba, n_bins=8)
    axes[0].plot(mean_pred, frac_pos, style, label=label)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibración perfecta')
axes[0].set_xlabel('Probabilidad predicha (Blue gana)')
axes[0].set_ylabel('Fracción de victorias reales')
axes[0].set_title('Reliability Diagram')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribución de probabilidades
axes[1].hist(raw_proba, bins=20, alpha=0.6, label='Sin calibrar', color='steelblue', edgecolor='white')
axes[1].hist(cal_proba, bins=20, alpha=0.6, label='Calibrado', color='darkorange', edgecolor='white')
axes[1].set_xlabel('Probabilidad predicha')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Probabilidades')
axes[1].legend()

plt.tight_layout()
plt.show()

# ROC curves de todos los modelos en fold 2
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models2.items():
    proba = model.predict_proba(X_r35)[:, 1]
    RocCurveDisplay.from_predictions(y_r35, proba, name=name, ax=ax)
RocCurveDisplay.from_predictions(y_r35, cal_proba, name=f'{best_model_name} Calibrado', ax=ax, linestyle='--')
ax.set_title('Curvas ROC — Fold 2 (Rounds 3-5)')
plt.tight_layout()
plt.show()

print(f"\n--- Métricas modelo calibrado en Fold 2 ---")
print(f"Log Loss: {log_loss(y_r35, cal_proba):.4f}  (menor = mejor para apuestas)")
print(f"Brier:    {brier_score_loss(y_r35, cal_proba):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_r35, cal_proba):.4f}")

---
## Sección 7 — Importancia de Features

In [ ]:
# Feature importance de XGBoost (Fold 2)
xgb_model = models2['XGBoost']
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(25)

plt.figure(figsize=(12, 8))
colors = ['#d9534f' if 'diff_' in f else '#5cb85c' if 'blue_' in f else '#5bc0de'
          for f in importance_df['feature']]
plt.barh(importance_df['feature'], importance_df['importance'], color=colors)
plt.xlabel('Importancia (XGBoost gain)')
plt.title('Top 25 Features más Importantes — XGBoost Fold 2\n'
          '🔴 Diferenciales  🟢 Blue team  🔵 Red team')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Logistic Regression: coeficientes (más interpretables)
lr_model = models2['Logistic Regression']
lr_clf = lr_model.named_steps['clf']
scaler = lr_model.named_steps['scaler']
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coef': lr_clf.coef_[0]
}).reindex(pd.Series(lr_clf.coef_[0]).abs().sort_values(ascending=False).index)
coef_df = coef_df.sort_values('coef', key=abs, ascending=True).tail(20)

plt.figure(figsize=(10, 7))
colors = ['#5cb85c' if v > 0 else '#d9534f' for v in coef_df['coef']]
plt.barh(coef_df['feature'], coef_df['coef'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coeficiente (positivo = favorece victoria Blue)')
plt.title('Top 20 Coeficientes — Logistic Regression\n(Más interpretable para decisiones de apuestas)')
plt.tight_layout()
plt.show()

---
## Sección 8 — Estrategia de Apuestas: Expected Value + Kelly Criterion

### Fórmulas clave:
- **Expected Value (EV):** `EV = p_modelo × (odds - 1) - (1 - p_modelo)`
- **Quarter-Kelly:** `f* = [(p × odds - 1) / (odds - 1)] × 0.25`

**Reglas de gestión:**
1. Solo apostar si EV > 3% (cubre incertidumbre del modelo + vig de la casa)
2. Quarter-Kelly (25% del Kelly completo) para reducir varianza
3. Cap máximo: nunca arriesgar más del 10% del bankroll
4. No apostar en primeras semanas post-patch
5. Comparar odds en múltiples casas (Pinnacle, Bet365, GG.bet)

In [ ]:
def calculate_ev(model_prob, decimal_odds):
    """Expected Value como fracción de la apuesta. EV>0 = apuesta con ventaja."""
    return model_prob * (decimal_odds - 1) - (1 - model_prob)

def kelly_fraction(model_prob, decimal_odds, multiplier=0.25, max_fraction=0.10):
    """Quarter-Kelly con cap máximo de seguridad."""
    b = decimal_odds - 1.0
    f_star = (model_prob * (b + 1) - 1) / b
    f_star = max(0.0, f_star)         # no apostar EV negativo
    f_star = min(f_star, max_fraction) # cap de seguridad
    return f_star * multiplier

# Simular odds de la casa (en producción: scraping de odds reales)
# Generamos odds plausibles: la casa usa las probabilidades reales + 5% de vig
np.random.seed(RANDOM_SEED)
n_games = len(cal_proba)
true_p = y_r35.values.astype(float)

# Odds simuladas con 5% de margen de la casa sobre las probabilidades del modelo
# (En realidad deberías usar odds reales de Pinnacle/Bet365)
noise = np.random.normal(0, 0.05, n_games)
house_prob_blue = np.clip(cal_proba + noise, 0.05, 0.95)
house_prob_blue = house_prob_blue / (house_prob_blue + (1 - house_prob_blue) * 0.95)  # +5% vig
sim_odds = 1.0 / house_prob_blue

# Calcular EV y Kelly para cada partida
EV_THRESHOLD = 0.03  # Solo apostar si ventaja > 3%

bet_analysis = pd.DataFrame({
    'gameid':       match_df.loc[mask_r35, 'gameid'].values,
    'blue_team':    match_df.loc[mask_r35, 'blue_team'].values,
    'red_team':     match_df.loc[mask_r35, 'red_team'].values,
    'model_p_blue': cal_proba,
    'house_odds':   sim_odds,
    'implied_p':    1.0 / sim_odds,
    'ev':           [calculate_ev(p, o) for p, o in zip(cal_proba, sim_odds)],
    'kelly':        [kelly_fraction(p, o) for p, o in zip(cal_proba, sim_odds)],
    'actual':       y_r35.values,
})
bet_analysis['should_bet'] = bet_analysis['ev'] > EV_THRESHOLD

positive_bets = bet_analysis[bet_analysis['should_bet']]
print(f"Total partidas analizadas: {len(bet_analysis)}")
print(f"Apuestas con EV > {EV_THRESHOLD:.0%}: {len(positive_bets)} ({100*len(positive_bets)/len(bet_analysis):.1f}%)")
if len(positive_bets) > 0:
    print(f"EV promedio en apuestas seleccionadas: {positive_bets['ev'].mean():.3f}")
    print(f"Kelly promedio: {positive_bets['kelly'].mean():.3f} ({positive_bets['kelly'].mean()*100:.1f}% del bankroll)")
    print(f"\nEjemplo de apuestas top:")
    print(positive_bets.nlargest(5, 'ev')[['blue_team','red_team','model_p_blue','house_odds','ev','kelly']].round(3).to_string(index=False))

In [ ]:
# Simulación de bankroll con Quarter-Kelly
def simulate_bankroll(bets_df, initial_bankroll=1000.0, ev_threshold=0.03):
    """
    Simula la evolución del bankroll apostando solo en partidas con EV > umbral.
    Retorna: historial de bankroll, estadísticas.
    """
    bankroll = initial_bankroll
    history  = [bankroll]
    wins, losses = 0, 0

    selected = bets_df[bets_df['ev'] > ev_threshold].copy()

    for _, row in selected.iterrows():
        stake = bankroll * row['kelly']
        if stake < 1.0:  # apuesta mínima
            history.append(bankroll)
            continue
        won = (row['actual'] == 1)
        if won:
            bankroll += stake * (row['house_odds'] - 1)
            wins += 1
        else:
            bankroll -= stake
            losses += 1
        bankroll = max(bankroll, 0)
        history.append(bankroll)

    return history, wins, losses, selected

history, wins, losses, selected_bets = simulate_bankroll(bet_analysis, initial_bankroll=1000.0)

# Gráfico de bankroll
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Simulación de Bankroll — Quarter-Kelly (EV > 3%)\n⚠️ Con ODDS SIMULADAS — reemplazar con odds reales para uso real', fontsize=12)

axes[0].plot(history, linewidth=2, color='steelblue')
axes[0].axhline(1000, ls='--', color='gray', alpha=0.5, label='Bankroll inicial')
axes[0].fill_between(range(len(history)), 1000, history,
                     where=[h >= 1000 for h in history], alpha=0.2, color='green', label='Ganancia')
axes[0].fill_between(range(len(history)), 1000, history,
                     where=[h < 1000 for h in history], alpha=0.2, color='red', label='Pérdida')
axes[0].set_xlabel('Apuesta #')
axes[0].set_ylabel('Bankroll ($)')
axes[0].set_title('Evolución del Bankroll')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribución de EV
axes[1].hist(bet_analysis['ev'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(EV_THRESHOLD, color='orange', ls='--', linewidth=2, label=f'Umbral EV={EV_THRESHOLD:.0%}')
axes[1].axvline(0, color='red', ls='-', linewidth=1, alpha=0.5, label='EV=0')
axes[1].set_xlabel('Expected Value')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de EV por Partida')
axes[1].legend()

plt.tight_layout()
plt.show()

# Resumen final
roi = (history[-1] - history[0]) / history[0] * 100
wr = wins / (wins + losses) * 100 if (wins + losses) > 0 else 0
print("=" * 55)
print("RESUMEN SIMULACIÓN DE APUESTAS")
print("=" * 55)
print(f"Partidas totales evaluadas: {len(bet_analysis)}")
print(f"Apuestas realizadas:        {wins + losses}")
print(f"Victorias / Derrotas:       {wins} / {losses} ({wr:.1f}% WR)")
print(f"Bankroll inicial:           $1,000.00")
print(f"Bankroll final:             ${history[-1]:,.2f}")
print(f"ROI:                        {roi:+.1f}%")
print("=" * 55)
print("\n⚠️  AVISO: Esta simulación usa ODDS SIMULADAS.")
print("Para uso real, necesitas scraping de odds de Pinnacle/Bet365.")

---
## Sección 9 — Guardar y Cargar el Modelo

In [ ]:
import joblib
import json

# Guardar modelo calibrado
joblib.dump(calibrated_model, 'lck_betting_model.pkl')

# Guardar también los nombres de features para usarlos al predecir nuevas partidas
with open('lck_feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print("Archivos guardados:")
print("  lck_betting_model.pkl  — modelo calibrado")
print("  lck_feature_cols.json  — lista de features")

# --- Para cargar y usar el modelo en el futuro ---
# model = joblib.load('lck_betting_model.pkl')
# with open('lck_feature_cols.json') as f:
#     cols = json.load(f)
#
# # X_nuevo debe tener las mismas columnas que feature_cols
# probabilidad_blue_gana = model.predict_proba(X_nuevo[cols])[:, 1]